In [1]:
"""
Copa do Mundo 2026 — Tracker de Predições
Coleta resultados reais via football-data.org e atualiza a planilha de predições.
"""
 
import requests
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from datetime import datetime, date
import sys
import os
import shutil

In [2]:
# ─────────────────────────────────────────────
# CONFIGURAÇÃO
# ─────────────────────────────────────────────
API_KEY = "55aebecc1b27424ca7ed3108e3b4940b"
API_BASE = "https://api.football-data.org/v4"
HEADERS = {"X-Auth-Token": API_KEY}
WC_COMPETITION = "WC"  # Código da Copa do Mundo no football-data.org
 
# Arquivo de entrada/saída
INPUT_FILE = r"C:\Users\raphael.eugenio\Desktop\Raphael\WC26\copa2026_predicoes_completas.xlsx"
OUTPUT_FILE = "copa2026_predicoes_atualizadas.xlsx"
 
# Colunas da aba "Predições - Fase Grupos" (índice 0-based)
COL_TIME1     = 5   # F — Time 1
COL_PLACAR    = 6   # G — Placar previsto
COL_TIME2     = 7   # H — Time 2
COL_DATA      = 2   # C — Data
COL_STATUS    = 20  # U — Status
COL_PLACAR_R  = 21  # V — Placar Real
COL_RESULT_R  = 22  # W — Resultado Real
COL_ACERTO_P  = 23  # X — Acerto Placar Exato
COL_ACERTO_R  = 24  # Y — Acerto Resultado
 
# Cores
GREEN  = PatternFill("solid", fgColor="C6EFCE")
RED    = PatternFill("solid", fgColor="FFC7CE")
YELLOW = PatternFill("solid", fgColor="FFEB9C")
BLUE   = PatternFill("solid", fgColor="BDD7EE")
GRAY   = PatternFill("solid", fgColor="D9D9D9")
 

In [3]:
# ─────────────────────────────────────────────
# MAPEAMENTO DE NOMES (PT → inglês da API)
# ─────────────────────────────────────────────
import unicodedata

TEAM_MAP = {
    "México": "Mexico",
    "África do Sul": "South Africa",
    "Coreia do Sul": "Korea Republic",
    "República Tcheca": "Czech Republic",
    "Canadá": "Canada",
    "Bósnia e Herzegovina": "Bosnia and Herzegovina",
    "Catar": "Qatar",
    "Suíça": "Switzerland",
    "Brasil": "Brazil",
    "Marrocos": "Morocco",
    "Haiti": "Haiti",
    "Escócia": "Scotland",
    "EUA": "USA",
    "Paraguai": "Paraguay",
    "Austrália": "Australia",
    "Turquia": "Türkiye",
    "Alemanha": "Germany",
    "Curaçao": "Curaçao",
    "Costa do Marfim": "Côte d'Ivoire",
    "Equador": "Ecuador",
    "Holanda": "Netherlands",
    "Japão": "Japan",
    "Suécia": "Sweden",
    "Tunísia": "Tunisia",
    "Bélgica": "Belgium",
    "Egito": "Egypt",
    "Irã": "Iran",
    "Nova Zelândia": "New Zealand",
    "Espanha": "Spain",
    "Cabo Verde": "Cape Verde",
    "Arábia Saudita": "Saudi Arabia",
    "Uruguai": "Uruguay",
    "França": "France",
    "Senegal": "Senegal",
    "Iraque": "Iraq",
    "Noruega": "Norway",
    "Argentina": "Argentina",
    "Argélia": "Algeria",
    "Áustria": "Austria",
    "Jordânia": "Jordan",
    "Portugal": "Portugal",
    "RD Congo": "DR Congo",
    "Uzbequistão": "Uzbekistan",
    "Colômbia": "Colombia",
    "Inglaterra": "England",
    "Croácia": "Croatia",
    "Gana": "Ghana",
    "Panamá": "Panama",
}

# Inverso: inglês → português
TEAM_MAP_INV = {v: k for k, v in TEAM_MAP.items()}

# Apelidos: como a API football-data.org pode escrever o nome → nome "oficial" do TEAM_MAP.
# (Esta é a causa nº1 de jogos que "não carregam": o nome da API não bate com o do mapa.)
# Tudo aqui é comparado já normalizado (sem acento, minúsculo), então cobre várias grafias.
API_ALIASES = {
    "united states": "USA",
    "usa": "USA",
    "south korea": "Korea Republic",
    "korea republic": "Korea Republic",
    "ivory coast": "Côte d'Ivoire",
    "cote d'ivoire": "Côte d'Ivoire",
    "turkey": "Türkiye",
    "turkiye": "Türkiye",
    "czechia": "Czech Republic",
    "ir iran": "Iran",
    "iran": "Iran",
    "cape verde islands": "Cape Verde",
    "dr congo": "DR Congo",
    "congo dr": "DR Congo",
    "netherlands": "Netherlands",
    "holland": "Netherlands",
    "bosnia-herzegovina": "Bosnia and Herzegovina",
    "bosnia and herzegovina": "Bosnia and Herzegovina",
}


def _sem_acento(texto: str) -> str:
    """Remove acentos: 'Côte d'Ivoire' → 'cote d'ivoire'."""
    nfkd = unicodedata.normalize("NFKD", texto)
    return "".join(c for c in nfkd if not unicodedata.combining(c))


def normalizar(nome: str) -> str:
    """Normaliza para comparação: sem acento, minúsculo, espaços colapsados."""
    if nome is None:
        return ""
    base = _sem_acento(str(nome)).strip().lower()
    return " ".join(base.split())


def nome_canonico(nome_api: str) -> str:
    """Resolve o nome vindo da API para a forma usada no TEAM_MAP, via aliases."""
    chave = normalizar(nome_api)
    if chave in API_ALIASES:
        return API_ALIASES[chave]
    return nome_api


def resultado_str(gols_t1: int, gols_t2: int, time1_pt: str, time2_pt: str) -> str:
    if gols_t1 > gols_t2:
        return f"Vitória {time1_pt}"
    elif gols_t1 < gols_t2:
        return f"Vitória {time2_pt}"
    return "Empate"


In [4]:
# ─────────────────────────────────────────────
# BUSCA DE PARTIDAS NA API
# ─────────────────────────────────────────────
def buscar_partidas() -> list[dict]:
    url = f"{API_BASE}/competitions/{WC_COMPETITION}/matches"
    print(f"🌐 Buscando partidas em {url} ...")
    resp = requests.get(url, headers=HEADERS, timeout=15)
    if resp.status_code == 200:
        dados = resp.json()
        partidas = dados.get("matches", [])
        print(f"   ✅ {len(partidas)} partidas recebidas da API.")
        return partidas
    elif resp.status_code == 404:
        print("   ⚠️  Competição WC 2026 ainda não disponível na API. Tentando busca alternativa...")
        return buscar_partidas_alternativo()
    else:
        print(f"   ❌ Erro HTTP {resp.status_code}: {resp.text[:200]}")
        return []


def buscar_partidas_alternativo() -> list[dict]:
    """Tenta buscar pelo ano corrente."""
    url = f"{API_BASE}/competitions/WC/matches?season=2026"
    resp = requests.get(url, headers=HEADERS, timeout=15)
    if resp.status_code == 200:
        dados = resp.json()
        partidas = dados.get("matches", [])
        print(f"   ✅ {len(partidas)} partidas recebidas (fallback).")
        return partidas
    print(f"   ❌ Fallback falhou ({resp.status_code}). Verifique sua chave API ou aguarde os jogos começarem.")
    return []


def _extrair_placar(m: dict):
    """Extrai (gols_home, gols_away). Tenta fullTime e, se vazio, regularTime."""
    score = m.get("score", {}) or {}
    for campo in ("fullTime", "regularTime"):
        bloco = score.get(campo) or {}
        gh, ga = bloco.get("home"), bloco.get("away")
        if gh is not None and ga is not None:
            return gh, ga
    return None, None


def indexar_partidas(partidas: list[dict]) -> dict:
    """Cria índice: (time1_canon, time2_canon) → resultado. Já resolve aliases da API."""
    idx: dict[tuple, dict] = {}
    finalizadas_log = []
    for m in partidas:
        status = m.get("status", "")
        if status not in ("FINISHED", "IN_PLAY", "PAUSED", "AWARDED"):
            continue
        ht = nome_canonico(m.get("homeTeam", {}).get("name", ""))
        at = nome_canonico(m.get("awayTeam", {}).get("name", ""))
        gh, ga = _extrair_placar(m)
        if gh is None or ga is None:
            continue
        idx[(normalizar(ht), normalizar(at))] = {"gols_home": gh, "gols_away": ga, "status": status}
        idx[(normalizar(at), normalizar(ht))] = {"gols_home": ga, "gols_away": gh, "status": status}
        finalizadas_log.append((ht, at, gh, ga, status))

    # 🔎 DIAGNÓSTICO: mostra tudo que a API considera já jogado/em andamento
    if finalizadas_log:
        print(f"\n   🔎 {len(finalizadas_log)} jogo(s) com placar vindo da API:")
        for ht, at, gh, ga, st in finalizadas_log:
            print(f"      • {ht} {gh}-{ga} {at}   [{st}]")
    else:
        print("\n   🔎 Nenhum jogo com placar disponível na API ainda.")
    return idx


def encontrar_resultado(idx: dict, time1_pt: str, time2_pt: str):
    """Tenta casar a linha da planilha com o índice da API por várias formas."""
    candidatos1 = {time1_pt, TEAM_MAP.get(time1_pt, time1_pt)}
    candidatos2 = {time2_pt, TEAM_MAP.get(time2_pt, time2_pt)}
    for c1 in candidatos1:
        for c2 in candidatos2:
            chave = (normalizar(c1), normalizar(c2))
            if chave in idx:
                return idx.get(chave)
    return None


In [5]:
# ─────────────────────────────────────────────
# ATUALIZAÇÃO DA PLANILHA
# ─────────────────────────────────────────────
def atualizar_planilha(partidas: list[dict]):
    if not os.path.exists(INPUT_FILE):
        print(f"❌ Arquivo '{INPUT_FILE}' não encontrado.")
        sys.exit(1)

    shutil.copy2(INPUT_FILE, OUTPUT_FILE)
    wb = load_workbook(OUTPUT_FILE)

    aba_nome = None
    for nome in wb.sheetnames:
        if "Predições" in nome or "Predicoes" in nome or "Fase Grupos" in nome:
            aba_nome = nome
            break

    if not aba_nome:
        print(f"❌ Aba de predições não encontrada. Abas disponíveis: {wb.sheetnames}")
        return

    ws = wb[aba_nome]
    idx = indexar_partidas(partidas)

    acertos_placar = 0
    acertos_resultado = 0
    jogos_finalizados = 0
    total_linhas = 0
    pares_casados = set()   # 🔎 guarda quais jogos da API foram efetivamente usados

    print(f"\n📋 Atualizando aba '{aba_nome}'...")

    for row in ws.iter_rows(min_row=2):
        time1_cell = row[COL_TIME1]
        time2_cell = row[COL_TIME2]
        placar_cell = row[COL_PLACAR]

        time1 = time1_cell.value
        time2 = time2_cell.value
        placar_prev = placar_cell.value if placar_cell else None

        if not time1 or not time2:
            continue

        total_linhas += 1
        resultado = encontrar_resultado(idx, str(time1), str(time2))

        status_cell    = row[COL_STATUS]
        placar_r_cell  = row[COL_PLACAR_R]
        result_r_cell  = row[COL_RESULT_R]
        acerto_p_cell  = row[COL_ACERTO_P]
        acerto_r_cell  = row[COL_ACERTO_R]

        if resultado is None:
            status_cell.value = "Agendado"
            status_cell.fill = GRAY
            placar_r_cell.value = None
            result_r_cell.value = None
            acerto_p_cell.value = None
            acerto_r_cell.value = None
            continue

        # registra o par casado (ambas as ordens) para o diagnóstico final
        t1_en = TEAM_MAP.get(str(time1), str(time1))
        t2_en = TEAM_MAP.get(str(time2), str(time2))
        pares_casados.add((normalizar(t1_en), normalizar(t2_en)))
        pares_casados.add((normalizar(t2_en), normalizar(t1_en)))

        jogos_finalizados += 1
        gh = resultado["gols_home"]
        ga = resultado["gols_away"]

        placar_real = f"{gh}-{ga}"
        result_real = resultado_str(gh, ga, str(time1), str(time2))

        placar_r_cell.value = placar_real
        result_r_cell.value = result_real

        acerto_placar = "✅" if placar_prev == placar_real else "❌"
        acerto_p_cell.value = acerto_placar

        result_prev = row[13].value if len(row) > 13 else None
        acerto_res = "✅" if result_prev and normalizar(str(result_prev)) == normalizar(result_real) else "❌"
        acerto_r_cell.value = acerto_res

        if acerto_placar == "✅":
            acertos_placar += 1
        if acerto_res == "✅":
            acertos_resultado += 1

        status_cell.value = "Finalizado"
        status_cell.fill = BLUE
        acerto_p_cell.fill = GREEN if acerto_placar == "✅" else RED
        acerto_r_cell.fill = GREEN if acerto_res == "✅" else RED

    # 🔎 DIAGNÓSTICO: jogos que a API trouxe mas NÃO casaram com nenhuma linha da planilha
    nao_casados = []
    for m in partidas:
        if m.get("status", "") not in ("FINISHED", "IN_PLAY", "PAUSED", "AWARDED"):
            continue
        ht = nome_canonico(m.get("homeTeam", {}).get("name", ""))
        at = nome_canonico(m.get("awayTeam", {}).get("name", ""))
        gh, ga = _extrair_placar(m)
        if gh is None or ga is None:
            continue
        if (normalizar(ht), normalizar(at)) not in pares_casados:
            nao_casados.append((m.get("homeTeam", {}).get("name", ""),
                                m.get("awayTeam", {}).get("name", ""), gh, ga))

    wb.save(OUTPUT_FILE)
    print(f"\n{'─'*50}")
    print(f"📊 RESUMO DA ATUALIZAÇÃO")
    print(f"{'─'*50}")
    print(f"   Total de linhas processadas : {total_linhas}")
    print(f"   Jogos já realizados         : {jogos_finalizados}")
    print(f"   Acertos de placar exato     : {acertos_placar}/{jogos_finalizados}")
    print(f"   Acertos de resultado (V/E/D): {acertos_resultado}/{jogos_finalizados}")
    if jogos_finalizados > 0:
        print(f"   Taxa acerto placar          : {acertos_placar / jogos_finalizados * 100:.1f}%")
        print(f"   Taxa acerto resultado       : {acertos_resultado / jogos_finalizados * 100:.1f}%")
    if nao_casados:
        print(f"\n   ⚠️  {len(nao_casados)} jogo(s) da API NÃO casaram com a planilha")
        print(f"      (provável nome divergente — adicione em TEAM_MAP ou API_ALIASES):")
        for ht, at, gh, ga in nao_casados:
            print(f"      • API: '{ht}' {gh}-{ga} '{at}'")
    print(f"{'─'*50}")
    print(f"✅ Arquivo salvo: {OUTPUT_FILE}")


In [6]:
# ─────────────────────────────────────────────
# DIAGNÓSTICO (modo --check)
# ─────────────────────────────────────────────
def verificar_api():
    print("🔍 Verificando acesso à API football-data.org...")
    url = f"{API_BASE}/competitions"
    resp = requests.get(url, headers=HEADERS, timeout=10)
    if resp.status_code == 200:
        comps = [c["code"] for c in resp.json().get("competitions", [])]
        wc_ok = "WC" in comps
        print(f"   ✅ API acessível. Copa do Mundo disponível: {'SIM ✅' if wc_ok else 'NÃO ❌ (aguardar liberação)'}")
        print(f"   Competições disponíveis: {', '.join(comps[:10])}...")
    else:
        print(f"   ❌ Erro {resp.status_code}: verifique a chave API.")
 
 

In [7]:
# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
if __name__ == "__main__":
    print("⚽  Copa do Mundo 2026 — Tracker de Predições")
    print(f"   Hora: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")
    print()
 
    if "--check" in sys.argv:
        verificar_api()
        sys.exit(0)
 
    partidas = buscar_partidas()
 
    if not partidas:
        print("\n⚠️  Nenhuma partida encontrada na API.")
        print("   Possíveis causas:")
        print("   • A Copa 2026 ainda não foi registrada na API (aguarde o início)")
        print("   • A chave API não tem permissão para WC")
        print("   • Execute com --check para diagnóstico")
        print("\n   A planilha será salva sem alterações de resultado real.")
        partidas = []
 
    atualizar_planilha(partidas)

⚽  Copa do Mundo 2026 — Tracker de Predições
   Hora: 12/06/2026 16:58:22

🌐 Buscando partidas em https://api.football-data.org/v4/competitions/WC/matches ...
   ✅ 104 partidas recebidas da API.

   🔎 3 jogo(s) com placar vindo da API:
      • Mexico 2-0 South Africa   [FINISHED]
      • Korea Republic 2-1 Czech Republic   [FINISHED]
      • Canada 0-1 Bosnia and Herzegovina   [PAUSED]

📋 Atualizando aba 'Predições - Fase Grupos'...

──────────────────────────────────────────────────
📊 RESUMO DA ATUALIZAÇÃO
──────────────────────────────────────────────────
   Total de linhas processadas : 72
   Jogos já realizados         : 3
   Acertos de placar exato     : 0/3
   Acertos de resultado (V/E/D): 2/3
   Taxa acerto placar          : 0.0%
   Taxa acerto resultado       : 66.7%
──────────────────────────────────────────────────
✅ Arquivo salvo: copa2026_predicoes_atualizadas.xlsx
